# Molecules as Data

In this notebook, we'll learn to represent molecules as data — strings, numbers, and images that a computer can process.

Just as climate data is temperature measurements over time, and audio data is amplitude samples over time, **molecular data is atoms and bonds encoded as text and numbers**.

We'll use:
- **SMILES** — a text notation for molecular structure
- **RDKit** — the standard cheminformatics toolkit
- **Molecular descriptors** — numbers that quantify chemical properties

## Step 1: Install & Import Libraries

In [ ]:
!pip install rdkit-pypi pubchempy -q

from rdkit import Chem
from rdkit.Chem import Draw, Descriptors, AllChem, DataStructs
from rdkit.Chem import PandasTools
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## Step 2: What Is SMILES?

**SMILES** (Simplified Molecular Input Line Entry System) is a way to write molecular structures as text strings. It's the "language" computers use to read molecules.

| SMILES | Molecule |
|--------|----------|
| `CCO` | Ethanol |
| `CC(=O)O` | Acetic acid |
| `c1ccccc1` | Benzene |
| `CC(=O)Oc1ccccc1C(=O)O` | Aspirin |
| `OC[C@@H](O1)[C@@H](O)[C@H](O)[C@@H](O)[C@@H]1O` | Glucose |

Atoms are letters, bonds are implied (single) or explicit (`=` double, `#` triple). Rings use matching numbers. Branches use parentheses.

In [ ]:
# Convert SMILES to RDKit molecule objects
smiles_list = {
    "Ethanol": "CCO",
    "Acetic Acid": "CC(=O)O",
    "Benzene": "c1ccccc1",
    "Aspirin": "CC(=O)Oc1ccccc1C(=O)O",
    "Caffeine": "Cn1c(=O)c2c(ncn2C)n(C)c1=O",
    "Glucose": "OC[C@@H](O1)[C@@H](O)[C@H](O)[C@@H](O)[C@@H]1O",
}

mols = {name: Chem.MolFromSmiles(s) for name, s in smiles_list.items()}

for name, mol in mols.items():
    print(f"{name:15s}  Atoms: {mol.GetNumAtoms():3d}  Bonds: {mol.GetNumBonds():3d}")

## Step 3: Drawing Molecules

RDKit can render 2D structure diagrams — the chemist's equivalent of a waveform or spectrogram.

In [ ]:
# Draw a grid of our molecules
mol_list = list(mols.values())
name_list = list(mols.keys())

img = Draw.MolsToGridImage(mol_list, molsPerRow=3, subImgSize=(300, 250),
                            legends=name_list)
img

In [ ]:
# Draw a single molecule with atom indices
from rdkit.Chem import rdMolDraw2D
from IPython.display import SVG

mol = mols["Aspirin"]
AllChem.Compute2DCoords(mol)

drawer = rdMolDraw2D.MolDraw2DSVG(400, 300)
drawer.drawOptions().addAtomIndices = True
drawer.DrawMolecule(mol)
drawer.FinishDrawing()
SVG(drawer.GetDrawingText())

## Step 4: Molecular Descriptors — Turning Structure into Numbers

Just as MFCCs turn audio into numbers and monthly means turn climate into numbers, **molecular descriptors** turn structure into numbers.

| Descriptor | What It Measures |
|-----------|-----------------|
| Molecular Weight (MW) | Size/mass of the molecule |
| LogP | Lipophilicity (fat vs. water solubility) |
| TPSA | Topological Polar Surface Area — drug absorption |
| NumHDonors | Hydrogen bond donors — intermolecular interactions |
| NumHAcceptors | Hydrogen bond acceptors |
| NumRotatableBonds | Flexibility of the molecule |
| NumAromaticRings | Aromatic ring count |
| FractionCSP3 | Fraction of carbons that are sp3 (3D character) |

In [ ]:
def compute_descriptors(mol):
    """Compute a set of molecular descriptors."""
    return {
        "MW": Descriptors.MolWt(mol),
        "LogP": Descriptors.MolLogP(mol),
        "TPSA": Descriptors.TPSA(mol),
        "HBD": Descriptors.NumHDonors(mol),
        "HBA": Descriptors.NumHAcceptors(mol),
        "RotBonds": Descriptors.NumRotatableBonds(mol),
        "AromaticRings": Descriptors.NumAromaticRings(mol),
        "FractionCSP3": Descriptors.FractionCSP3(mol),
        "NumAtoms": mol.GetNumHeavyAtoms(),
    }

# Compute for all molecules
desc_data = {}
for name, mol in mols.items():
    desc_data[name] = compute_descriptors(mol)

desc_df = pd.DataFrame(desc_data).T
desc_df

## Step 5: Visualizing Molecular Properties

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

names = desc_df.index.tolist()
colors = ["#e74c3c", "#3498db", "#2ecc71", "#f39c12", "#9b59b6", "#1abc9c"]

# MW bar chart
axes[0].barh(names, desc_df["MW"], color=colors)
axes[0].set_xlabel("Molecular Weight (g/mol)")
axes[0].set_title("Molecular Weight")

# LogP bar chart
axes[1].barh(names, desc_df["LogP"], color=colors)
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set_xlabel("LogP")
axes[1].set_title("LogP (Lipophilicity)")

# TPSA bar chart
axes[2].barh(names, desc_df["TPSA"], color=colors)
axes[2].set_xlabel("TPSA (Å²)")
axes[2].set_title("Polar Surface Area")

plt.tight_layout()
plt.show()

print("LogP > 0: lipophilic (prefers fat/oil). LogP < 0: hydrophilic (prefers water).")
print("High TPSA: poor membrane permeability. Low TPSA: good absorption.")

## Step 6: Molecular Fingerprints — The Chemical "Spectrogram"

A **molecular fingerprint** is a binary vector (list of 0s and 1s) that encodes which structural features are present in a molecule. It's like a spectrogram — a pattern that captures the molecule's identity.

We'll use **Morgan fingerprints** (ECFP), the most widely used in cheminformatics.

In [ ]:
# Generate Morgan fingerprints (radius=2, 1024 bits)
fps = {}
for name, mol in mols.items():
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=1024)
    arr = np.zeros(1024)
    DataStructs.ConvertToNumpyArray(fp, arr)
    fps[name] = arr

# Visualize as a heatmap — each row is a molecule, each column is a feature bit
fp_matrix = np.array(list(fps.values()))

plt.figure(figsize=(15, 4))
plt.imshow(fp_matrix, aspect="auto", cmap="Blues", interpolation="nearest")
plt.yticks(range(len(fps)), list(fps.keys()))
plt.xlabel("Fingerprint Bit Index (1024 bits)")
plt.title("Morgan Fingerprints — Molecular 'Spectrograms'")
plt.colorbar(label="Bit On/Off")
plt.show()

print(f"Each molecule is represented by {fp_matrix.shape[1]} binary features.")
print("Bright = structural feature present. Dark = absent.")

## Step 7: Molecular Similarity

Fingerprints let us compute **similarity** between molecules using the **Tanimoto coefficient** (like cosine similarity for binary vectors).

Tanimoto = 1.0 means identical fingerprints. Tanimoto = 0.0 means no shared features.

In [ ]:
# Compute pairwise Tanimoto similarity
names = list(mols.keys())
n = len(names)
sim_matrix = np.zeros((n, n))

fp_objects = [AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=1024) for mol in mols.values()]

for i in range(n):
    for j in range(n):
        sim_matrix[i, j] = DataStructs.TanimotoSimilarity(fp_objects[i], fp_objects[j])

plt.figure(figsize=(7, 6))
plt.imshow(sim_matrix, cmap="YlOrRd", vmin=0, vmax=1)
plt.xticks(range(n), names, rotation=45, ha="right")
plt.yticks(range(n), names)
plt.colorbar(label="Tanimoto Similarity")
plt.title("Pairwise Molecular Similarity")

# Annotate with values
for i in range(n):
    for j in range(n):
        plt.text(j, i, f"{sim_matrix[i,j]:.2f}", ha="center", va="center", fontsize=8)

plt.tight_layout()
plt.show()

## What's Next

In **Notebook 2**, we'll explore a larger library of compounds — comparing structural families, identifying patterns, and building intuition for chemical similarity, just as climate students compare city temperature profiles.